# 003 Unified Comparison of Non-DL Spatial Allocation Methods

Unified evaluation of 27 non-deep-learning spatial load allocation methods, covering:

**Baselines (3)**: ITL2_average, ITL2_to_ITL3_equal, ITL2_to_ITL3_area

**Euclidean distance + demand weighting (4)**: voronoi, civd, voronoi_gpm, civd_gpm

**WorldCover correction (2)**: voronoi_wc_gpm, civd_wc_gpm

**NTL nighttime-light correction (5)**: voronoi_ntl, civd_ntl, voronoi_ntl_gpm, civd_ntl_gpm, voronoi_wc_ntl_gpm

**Substation proximity correction (8)**:
- γ=1: voronoi_prox1_ntl, civd_prox1_ntl, voronoi_prox1_ntl_gpm, civd_prox1_ntl_gpm
- γ=2: voronoi_prox2_ntl, civd_prox2_ntl, voronoi_prox2_ntl_gpm, civd_prox2_ntl_gpm

**Road-network distance (4, conditional)**: voronoi_ND, civd_ND, voronoi_gpm_ND, civd_gpm_ND

**Oracle (1)**: ITL3_average

In [ ]:
import sys
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error, mean_absolute_error

PROJECT_ROOT = Path('../../').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from SpatialAllocation.Allocator import allocator_registry
from SpatialAllocation.Allocator.clustering.do_clustering import do_clustering
from SpatialAllocation.Weighter import weighter_registry
from SpatialAllocation.FeatureExtractor.correctors import corrector_registry
from SpatialAllocation.FeatureExtractor.correctors.proximity_corrector import ProximityCorrector

try:
    from SpatialAllocation.utils.NetworkDistance import load_distance_results
    _nd_import_ok = True
except ImportError:
    _nd_import_ok = False

warnings.filterwarnings('ignore', category=FutureWarning)

# ─── Path constants ───
DATA_DIR = Path('./results/intermediate')
ASSEMBLED_DIR = DATA_DIR / 'features' / 'assembled'
EXTRACTED_DIR = DATA_DIR / 'features' / 'extracted'
ND_DATA_DIR = DATA_DIR / 'features' / 'network_distance'
CIVD_CACHE_DIR = Path('./results/static_allocation')
ND_CIVD_CACHE_DIR = Path('./results/static_allocation_nd')
OUTPUT_DIR = Path('./results/static_allocation')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ─── Parameters ───
GAMMA_VALUES = [1.0, 2.0]
TARGET_CRS = 'EPSG:27700'
DIST_CLAMP_KM = 0.01

STUDY_REGIONS = [
    'London',
    'TLH2', 'TLH3', 'TLJ1', 'TLF1', 'TLF2',
    'TLC1', 'TLC2', 'TLD6', 'TLG1', 'TLG2', 'TLE4',
    'TLH1', 'TLE3', 'TLD3', 'TLD4',
]

LANDUSE_PERCENT_MAP = {
    'lu_residential_prop': 'residential_percent',
    'lu_commercial_prop': 'commercial_percent',
    'lu_industrial_prop': 'industrial_percent',
    'lu_agricultural_prop': 'agricultural_percent',
    'lu_others_prop': 'others_percent',
}
LU_COLS = list(LANDUSE_PERCENT_MAP.keys())
PCT_COLS = list(LANDUSE_PERCENT_MAP.values())

In [ ]:
# ─── Load data ───
region_gdf = gpd.read_file(str(DATA_DIR / 'ITL3_region.gpkg'))
substations_gdf = gpd.read_file(str(DATA_DIR / 'substations.gpkg'))

grids = {}
ntl_data = {}
nd_available = {}

for loc in STUDY_REGIONS:
    path = ASSEMBLED_DIR / f'{loc}_grid_points.pickle'
    with open(path, 'rb') as f:
        grid_gdf, step_size_m = pickle.load(f)
    grids[loc] = (grid_gdf, step_size_m)

    ntl_path = EXTRACTED_DIR / f'{loc}_ntl.npz'
    if ntl_path.exists():
        ntl_npz = np.load(ntl_path, allow_pickle=True)
        ntl_data[loc] = ntl_npz['data'][:, 0]
        assert len(ntl_data[loc]) == len(grid_gdf)

    nd_dir = ND_DATA_DIR / loc
    nd_available[loc] = nd_dir.exists() and _nd_import_ok

ntl_count = len(ntl_data)
nd_count = sum(nd_available.values())
print(f'Loaded {len(grids)} regions, NTL {ntl_count}/{len(grids)}, ND {nd_count}/{len(grids)}')

In [ ]:
# ─── Helper functions ───

def compute_demand(grid_gdf, region_sub, weighter_result, demand_col='demand'):
    W = weighter_result.weights
    gdf = grid_gdf.copy()
    gdf[demand_col] = 0.0
    region_info = region_sub.set_index('ITL3')
    for itl3, group in gdf.groupby('ITL3'):
        if itl3 not in region_info.index:
            continue
        total_demand = region_info.loc[itl3, 'Demand (MVA)']
        idx = group.index
        if W.ndim == 2:
            pcts = np.array([region_info.loc[itl3, c] for c in PCT_COLS])
            score = W[idx] @ pcts
        else:
            score = W[idx]
        score_sum = score.sum()
        if score_sum > 0:
            gdf.loc[idx, demand_col] = total_demand * score / score_sum
        else:
            gdf.loc[idx, demand_col] = total_demand / len(group)
    return gdf


def aggregate_to_substations(grid_gdf, subs_gdf, assignment, demand_col):
    result = subs_gdf.copy()
    result['allocated_demand'] = 0.0
    demands = grid_gdf[demand_col].values
    for target_idx in range(len(subs_gdf)):
        mask = assignment == target_idx
        result.loc[target_idx, 'allocated_demand'] = demands[mask].sum()
    return result


def aggregate_clustered_to_substations(grid_gdf, subs_gdf, cluster_gdf,
                                        assignment, demand_col):
    result = subs_gdf.copy()
    result['allocated_demand'] = 0.0
    demands = grid_gdf[demand_col].values
    cluster_demands = {}
    for label in np.unique(assignment):
        mask = assignment == label
        cluster_demands[label] = demands[mask].sum()
    for label, total_d in cluster_demands.items():
        members = cluster_gdf[cluster_gdf['cluster_label'] == label].index
        n_members = len(members)
        if n_members > 0:
            for idx in members:
                if idx < len(result):
                    result.loc[idx, 'allocated_demand'] += total_d / n_members
    return result


def evaluate_allocation(subs_result, actual_col='Demand (MVA)',
                        alloc_col='allocated_demand'):
    actual = subs_result[actual_col].values
    allocated = subs_result[alloc_col].values
    corr, _ = pearsonr(actual, allocated)
    rmse = np.sqrt(mean_squared_error(actual, allocated))
    mae = mean_absolute_error(actual, allocated)
    return {'corr': corr, 'rmse': rmse, 'mae': mae}


def reconstruct_full_nd_matrix(nd_matrix, nd_target_indices, n_targets):
    n_agents, k = nd_matrix.shape
    full = np.full((n_agents, n_targets), np.inf, dtype=np.float64)
    rows = np.repeat(np.arange(n_agents), k)
    cols = nd_target_indices.ravel()
    vals = nd_matrix.ravel()
    valid = cols >= 0
    full[rows[valid], cols[valid]] = vals[valid]
    return full

In [ ]:
METHOD_ORDER = [
    # baselines
    'ITL2_average', 'ITL2_to_ITL3_equal', 'ITL2_to_ITL3_area',
    # Euclidean + no correction
    'voronoi', 'civd', 'voronoi_gpm', 'civd_gpm',
    # WC
    'voronoi_wc_gpm', 'civd_wc_gpm',
    # NTL
    'voronoi_ntl', 'civd_ntl',
    'voronoi_ntl_gpm', 'civd_ntl_gpm',
    'voronoi_wc_ntl_gpm',
    # Proximity γ=1
    'voronoi_prox1_ntl', 'civd_prox1_ntl',
    'voronoi_prox1_ntl_gpm', 'civd_prox1_ntl_gpm',
    # Proximity γ=2
    'voronoi_prox2_ntl', 'civd_prox2_ntl',
    'voronoi_prox2_ntl_gpm', 'civd_prox2_ntl_gpm',
    # ND (conditional)
    'voronoi_ND', 'civd_ND',
    'voronoi_gpm_ND', 'civd_gpm_ND',
    # Oracle
    'ITL3_average',
]

In [ ]:
# ─── Main loop ───

wc_corrector = corrector_registry.create('wc')
ntl_corrector = corrector_registry.create('ntl')
prox_corrector = corrector_registry.create('proximity')

all_metrics = {}

for i, loc in enumerate(STUDY_REGIONS):
    grid_gdf, step_size_m = grids[loc]
    ntl_values = ntl_data[loc]
    study_itl3 = grid_gdf['ITL3'].unique()
    region_sub = region_gdf[region_gdf['ITL3'].isin(study_itl3)].copy()
    subs_sub = substations_gdf[substations_gdf['ITL3'].isin(study_itl3)].copy().reset_index(drop=True)

    # ── Base demand ──
    uniform = weighter_registry.create('uniform', config={})
    uniform_res = uniform.compute(grid_gdf, target_gdf=subs_sub)
    grid_gdf = compute_demand(grid_gdf, region_sub, uniform_res, demand_col='average_demand')

    gpm = weighter_registry.create('gpm', config={
        'mode': 'categorical', 'proportion_columns': LU_COLS,
    })
    gpm_res = gpm.compute(grid_gdf, target_gdf=subs_sub)
    grid_gdf = compute_demand(grid_gdf, region_sub, gpm_res, demand_col='landuse_demand')

    # ── WC correction ──
    wc_corrector.correct(grid_gdf, region_sub, 'landuse_demand',
                         grid_gdf['wc_others_ratio'].values, 'wc_landuse_demand')

    # ── NTL correction ──
    ntl_corrector.correct(grid_gdf, region_sub, 'average_demand',
                          ntl_values, 'ntl_average_demand')
    ntl_corrector.correct(grid_gdf, region_sub, 'landuse_demand',
                          ntl_values, 'ntl_landuse_demand')
    ntl_corrector.correct(grid_gdf, region_sub, 'wc_landuse_demand',
                          ntl_values, 'wc_ntl_landuse_demand')

    # ── Proximity correction ──
    for gamma in GAMMA_VALUES:
        g_tag = f'prox{int(gamma)}'
        prox_scores = ProximityCorrector.compute_scores(
            grid_gdf, subs_sub, gamma=gamma,
            target_crs=TARGET_CRS, clamp_km=DIST_CLAMP_KM)
        prox_corrector.correct(grid_gdf, region_sub, 'ntl_average_demand',
                               prox_scores, f'{g_tag}_ntl_average_demand')
        prox_corrector.correct(grid_gdf, region_sub, 'ntl_landuse_demand',
                               prox_scores, f'{g_tag}_ntl_landuse_demand')

    # ── Demand conservation check ──
    total_demand = region_sub['Demand (MVA)'].sum()
    check_cols = [
        'average_demand', 'landuse_demand', 'wc_landuse_demand',
        'ntl_average_demand', 'ntl_landuse_demand', 'wc_ntl_landuse_demand',
        'prox1_ntl_average_demand', 'prox1_ntl_landuse_demand',
        'prox2_ntl_average_demand', 'prox2_ntl_landuse_demand',
    ]
    for col in check_cols:
        assert abs(grid_gdf[col].sum() - total_demand) < 1.0, f'{loc}: {col}'

    # ── Voronoi allocation (Euclidean) ──
    results = {}
    alloc = allocator_registry.create('voronoi')
    voronoi_res = alloc.allocate(grid_gdf, subs_sub.copy())

    voronoi_methods = [
        ('voronoi', 'average_demand'),
        ('voronoi_gpm', 'landuse_demand'),
        ('voronoi_wc_gpm', 'wc_landuse_demand'),
        ('voronoi_ntl', 'ntl_average_demand'),
        ('voronoi_ntl_gpm', 'ntl_landuse_demand'),
        ('voronoi_wc_ntl_gpm', 'wc_ntl_landuse_demand'),
        ('voronoi_prox1_ntl', 'prox1_ntl_average_demand'),
        ('voronoi_prox1_ntl_gpm', 'prox1_ntl_landuse_demand'),
        ('voronoi_prox2_ntl', 'prox2_ntl_average_demand'),
        ('voronoi_prox2_ntl_gpm', 'prox2_ntl_landuse_demand'),
    ]
    for method, dcol in voronoi_methods:
        results[method] = aggregate_to_substations(
            grid_gdf, subs_sub, voronoi_res.assignment, dcol)

    # ── CIVD allocation (Euclidean, cached) ──
    coords = np.column_stack([subs_sub.geometry.x.values, subs_sub.geometry.y.values])
    cluster_gdf, centroid_gdf = do_clustering(coords, method='hdbscan', min_cluster_size=2)

    target_civd = subs_sub.copy()
    target_civd['cluster_label'] = cluster_gdf['cluster_label']

    civd_config = {
        'solver': 'scip', 'method': 'civd',
        'cluster_label_column': 'cluster_label', 'n_jobs': -1,
    }

    civd_cache_path = CIVD_CACHE_DIR / f'{loc}_civd_cache.pickle'
    civd_cache_valid = False
    if civd_cache_path.exists():
        with open(civd_cache_path, 'rb') as f:
            cache = pickle.load(f)
        if (cache.get('n_grid') == len(grid_gdf)
            and cache.get('n_target') == len(target_civd)
            and cache.get('config') == civd_config):
            civd_assignment = cache['assignment']
            civd_cache_valid = True

    if not civd_cache_valid:
        alloc_civd = allocator_registry.create('civd', config=civd_config)
        civd_res = alloc_civd.allocate(grid_gdf, target_civd)
        civd_assignment = civd_res.assignment
        with open(civd_cache_path, 'wb') as f:
            pickle.dump({
                'assignment': civd_assignment, 'n_grid': len(grid_gdf),
                'n_target': len(target_civd), 'config': civd_config,
            }, f)

    civd_methods = [
        ('civd', 'average_demand'),
        ('civd_gpm', 'landuse_demand'),
        ('civd_wc_gpm', 'wc_landuse_demand'),
        ('civd_ntl', 'ntl_average_demand'),
        ('civd_ntl_gpm', 'ntl_landuse_demand'),
        ('civd_prox1_ntl', 'prox1_ntl_average_demand'),
        ('civd_prox1_ntl_gpm', 'prox1_ntl_landuse_demand'),
        ('civd_prox2_ntl', 'prox2_ntl_average_demand'),
        ('civd_prox2_ntl_gpm', 'prox2_ntl_landuse_demand'),
    ]
    for method, dcol in civd_methods:
        results[method] = aggregate_clustered_to_substations(
            grid_gdf, subs_sub, cluster_gdf, civd_assignment, dcol)

    # ── ND allocation (conditional) ──
    if nd_available.get(loc, False):
        nd_dir = ND_DATA_DIR / loc
        nd_matrix, nd_target_idx, _, nd_target_map = load_distance_results(str(nd_dir))
        full_nd = reconstruct_full_nd_matrix(nd_matrix, nd_target_idx, len(subs_sub))

        nd_assignment = full_nd.argmin(axis=1)
        results['voronoi_ND'] = aggregate_to_substations(
            grid_gdf, subs_sub, nd_assignment, 'average_demand')
        results['voronoi_gpm_ND'] = aggregate_to_substations(
            grid_gdf, subs_sub, nd_assignment, 'landuse_demand')

        nd_civd_cache = ND_CIVD_CACHE_DIR / f'{loc}_civd_nd_cache.pickle'
        nd_civd_valid = False
        if nd_civd_cache.exists():
            with open(nd_civd_cache, 'rb') as f:
                cache = pickle.load(f)
            if (cache.get('n_grid') == len(grid_gdf)
                and cache.get('n_target') == len(target_civd)
                and cache.get('config') == civd_config):
                nd_civd_assignment = cache['assignment']
                nd_civd_valid = True

        if not nd_civd_valid:
            alloc_nd = allocator_registry.create('civd', config=civd_config)
            nd_civd_res = alloc_nd.allocate(grid_gdf, target_civd, distance_matrix=full_nd)
            nd_civd_assignment = nd_civd_res.assignment

        results['civd_ND'] = aggregate_clustered_to_substations(
            grid_gdf, subs_sub, cluster_gdf, nd_civd_assignment, 'average_demand')
        results['civd_gpm_ND'] = aggregate_clustered_to_substations(
            grid_gdf, subs_sub, cluster_gdf, nd_civd_assignment, 'landuse_demand')

    # ── Baseline methods ──
    itl2_demands = region_sub.groupby('ITL2')['Demand (MVA)'].sum()

    itl3_avg = subs_sub.copy()
    itl3_avg['allocated_demand'] = 0.0
    for itl3, group in itl3_avg.groupby('ITL3'):
        region_row = region_sub[region_sub['ITL3'] == itl3]
        if not region_row.empty:
            itl3_avg.loc[group.index, 'allocated_demand'] = \
                region_row['Demand (MVA)'].iloc[0] / len(group)
    results['ITL3_average'] = itl3_avg

    itl2_avg = subs_sub.copy()
    itl2_avg['allocated_demand'] = 0.0
    for itl2, group in itl2_avg.groupby('ITL2'):
        if itl2 in itl2_demands.index:
            itl2_avg.loc[group.index, 'allocated_demand'] = \
                itl2_demands[itl2] / len(group)
    results['ITL2_average'] = itl2_avg

    itl2_to_itl3_equal = subs_sub.copy()
    itl2_to_itl3_equal['allocated_demand'] = 0.0
    itl2_itl3_counts = region_sub.groupby('ITL2')['ITL3'].count()
    for itl3, group in itl2_to_itl3_equal.groupby('ITL3'):
        region_row = region_sub[region_sub['ITL3'] == itl3]
        if not region_row.empty:
            itl2 = region_row['ITL2'].iloc[0]
            itl3_demand = itl2_demands[itl2] / itl2_itl3_counts[itl2]
            itl2_to_itl3_equal.loc[group.index, 'allocated_demand'] = \
                itl3_demand / len(group)
    results['ITL2_to_ITL3_equal'] = itl2_to_itl3_equal

    itl2_to_itl3_area = subs_sub.copy()
    itl2_to_itl3_area['allocated_demand'] = 0.0
    itl2_area_totals = region_sub.groupby('ITL2')['area'].sum()
    for itl3, group in itl2_to_itl3_area.groupby('ITL3'):
        region_row = region_sub[region_sub['ITL3'] == itl3]
        if not region_row.empty:
            itl2 = region_row['ITL2'].iloc[0]
            area_ratio = region_row['area'].iloc[0] / itl2_area_totals[itl2]
            itl3_demand = itl2_demands[itl2] * area_ratio
            itl2_to_itl3_area.loc[group.index, 'allocated_demand'] = \
                itl3_demand / len(group)
    results['ITL2_to_ITL3_area'] = itl2_to_itl3_area

    # ── Evaluation ──
    metrics = {}
    for method_name, subs_result in results.items():
        metrics[method_name] = evaluate_allocation(subs_result)
    all_metrics[loc] = metrics

    print(f'  [{i+1}/{len(STUDY_REGIONS)}] {loc} done')

print(f'\n{len(STUDY_REGIONS)} regions complete')

In [ ]:
# ─── RMSE summary table ───
rmse_data = {}
for loc, metrics in all_metrics.items():
    rmse_data[loc] = {m: metrics[m]['rmse'] for m in metrics}

rmse_df = pd.DataFrame(rmse_data)
available = [m for m in METHOD_ORDER if m in rmse_df.index]
rmse_df = rmse_df.loc[available].round(2)
rmse_df.index.name = 'method'
rmse_df.columns.name = 'region'

display(rmse_df)
rmse_df.to_csv(OUTPUT_DIR / 'all_regions_rmse.csv')

In [ ]:
# ─── MAE summary table ───
mae_data = {}
for loc, metrics in all_metrics.items():
    mae_data[loc] = {m: metrics[m]['mae'] for m in metrics}

mae_df = pd.DataFrame(mae_data)
available = [m for m in METHOD_ORDER if m in mae_df.index]
mae_df = mae_df.loc[available].round(2)
mae_df.index.name = 'method'
mae_df.columns.name = 'region'

display(mae_df)
mae_df.to_csv(OUTPUT_DIR / 'all_regions_mae.csv')

In [ ]:
# ─── Correlation summary table ───
corr_data = {}
for loc, metrics in all_metrics.items():
    corr_data[loc] = {m: metrics[m]['corr'] for m in metrics}

corr_df = pd.DataFrame(corr_data)
available = [m for m in METHOD_ORDER if m in corr_df.index]
corr_df = corr_df.loc[available].round(4)
corr_df.index.name = 'method'
corr_df.columns.name = 'region'

display(corr_df)
corr_df.to_csv(OUTPUT_DIR / 'all_regions_corr.csv')